In [ ]:
from datetime import datetime
from dateutil.relativedelta import relativedelta
import json
from typing import Dict, List

import boto3
import botocore
import requests

#---
#1. get taxi data DONE
#2. get weather data DONE
#3. upload to S3
#4. organize the code
#5. create trigger for Lambda
#---
#Konstansok:
S3_BUCKET = "cubix-chichago-taxi-bb-v2"
TAXI_API_URL = "https://data.cityofchicago.org/api/v3/views/ajtu-isnz/query.json"
WEATHER_API_URL = "https://archive-api.open-meteo.com/v1/era5"

s3_cllient = boto3.client("s3")


def _get_data_from_api(url:str,params:Dict=None)->List[Dict]:
    #---
    #retrieves data from the API with parameters

    #:params: url
    #:params: params(optional)
    #:return: a list of dictionarities containing the data
    #---
    print(f"Retrieving data from {url}")
    response=requests.get(url,params=params)
    return response.json()


def get_taxi_data(date_str: str)->List[Dict]:
    #---
    #retrieves taxi data from the API
    
    #:params: date_str
    #:return: a list of dictionarities containing the data
    #---
    print(f"Retrieving taxi data from {date_str}")
    query=f"?app_token=ogjssL0qdXCJ16jANDq5lFDLc&query=select * where trip_start_timestamp>='{date_str}T00:00:00' AND trip_start_timestamp<='{date_str}T01:59:59'"
    return _get_data_from_api(TAXI_API_URL + query)


def get_weather_data(date_str: str)->List[Dict]:
    #---
    #retrieves weather data from the API

    #:params: date_str
    #:return: a list of dictionarities containing the data
    #---
    print(f"Retrieving weather data from {date_str}")
    params={    
        "latitude": 41.85,
        "longitude": -87.65,
        "start_date":date_str,
        "end_date":date_str,
        "hourly":"temperature_2m,wind_speed_10m,rain,precipitation"
    }
    return _get_data_from_api(WEATHER_API_URL, params=params)


def upload_to_s3(data: Dict, folder: str, filename: str) -> None:
    #---
    #uploads data to S3

    #:params data:          the data to upload
    #:params folder:        the folder to upload the data to
    #:params filename:      the filename to upload the data to
    #:raise ValueError:     if data None or empty
    #:raise RuntimeError:   error during uploading to S3
    #---
    print(f"Uploading {filename} to S3")
    if not data:
        raise ValueError(f"Data for {filename} is empty, stopping execution")

    s3_key = f"raw_data/to_processed/{folder}/{filename}"

    try:
        s3_cllient.put_object(
        Body=json.dumps(data),
        Bucket=S3_BUCKET,
        Key=s3_key
        )   
        print(f"Uploaded {filename} to S3")
    except Exception as e:
        raise RuntimeError(f"Error uploading {filename} to S3: {e}")


def lambda_handler(event, context):
    #---
    #Lambda function to retrieve taxi and weather data from the given date and upload it to S3

    #Steps:
    #   1.creadte the date which is today minus 2 months
    #   2.get the taxi raw data
    #   3.get the weather raw data
    #   4.upload them to S3
    #---
    date_str=(datetime.now() - relativedelta(months=2)).strftime("%Y-%m-%d")

    taxi_data=get_taxi_data(date_str)
    weather_data=get_weather_data(date_str)

    upload_to_s3(taxi_data, "taxi", f"taxi_{date_str}.json")
    upload_to_s3(weather_data, "weather", f"weather_{date_str}.json")


 #   print(taxi_data)
 #   print(weather_data)